# DefAPI Qwen2.5-Coder-14B LoRA Fine-tuning

This notebook fine-tunes `Qwen/Qwen2.5-Coder-14B-Instruct` for DefAPI security remediation responses using bf16 LoRA training in an A100 80GB RunPod Jupyter environment.

Run this notebook from a fresh RunPod Jupyter kernel. The first code cell installs a locked CUDA 12.4 package stack from `requirements-runpod-cu124.txt` before `torch` is imported, which avoids common Transformers / PEFT version conflicts.

The target assistant response format is:

1. 취약점 설명
2. 안전한 수정 코드
3. 수정 이유
4. 추가 주의사항

The default configuration is an A100 80GB full LoRA run: `hitoshura25/crossvul`, full `train` split, sequence length `2048`, W&B enabled, and output under `/workspace/checkpoints`.


## 1. Locked RunPod Package Install

Run this cell before any `torch`, `transformers`, or `peft` import. It installs only when the current kernel does not match the locked RunPod CUDA 12.4 stack.


In [15]:
import importlib.metadata
import os
import subprocess
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "requirements-runpod-cu124.txt").exists():
            return candidate
    raise FileNotFoundError("requirements-runpod-cu124.txt was not found from the current notebook directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

REQUIREMENTS_PATH = PROJECT_ROOT / "requirements-runpod-cu124.txt"
EXPECTED_VERSIONS = [
    ("torch", "2.5.1+cu124"),
    ("transformers", "4.46.3"),
    ("accelerate", "1.1.1"),
    ("datasets", "3.1.0"),
    ("peft", "0.13.2"),
    ("huggingface-hub", "0.26.5"),
    ("tokenizers", "0.20.3"),
    ("safetensors", "0.4.5"),
    ("sentencepiece", "0.2.0"),
    ("protobuf", "5.28.3"),
    ("PyYAML", "6.0.2"),
    ("wandb", "0.18.7"),
    ("packaging", "24.2"),
]


def installed_version(distribution: str) -> str | None:
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None


mismatches = []
for distribution, expected_version in EXPECTED_VERSIONS:
    current_version = installed_version(distribution)
    if current_version != expected_version:
        mismatches.append((distribution, current_version, expected_version))

if mismatches:
    print("Installing locked RunPod package stack:")
    for distribution, current_version, expected_version in mismatches:
        print(f"- {distribution}: {current_version or 'missing'} -> {expected_version}")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--no-cache-dir",
        "-r",
        str(REQUIREMENTS_PATH),
    ])
else:
    print("Locked RunPod package stack is already installed.")

print(f"Project root: {PROJECT_ROOT}")


Locked RunPod package stack is already installed.
Project root: /workspace


## 2. Environment Check


In [ ]:
import os
import platform
import subprocess
import sys
from pathlib import Path

print(f"Python: {sys.version}")
print(f"Executable: {sys.executable}")
print(f"Platform: {platform.platform()}")
print(f"Working directory: {Path.cwd()}")

try:
    import torch
    print(f"torch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA runtime: {torch.version.cuda}")
        for index in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(index)
            total_gb = props.total_memory / 1024**3
            print(f"GPU {index}: {props.name} ({total_gb:.1f} GB VRAM)")
    else:
        print("GPU: not available")
except (ImportError, RuntimeError) as exc:
    print(f"torch check failed: {exc}")

try:
    result = subprocess.run(["nvidia-smi"], check=False, text=True, capture_output=True)
    print(result.stdout[:2000] if result.stdout else "nvidia-smi produced no stdout")
except FileNotFoundError:
    print("nvidia-smi not found")

Python: 3.11.10 (main, Sep  7 2024, 18:35:41) [GCC 11.4.0]
Executable: /usr/bin/python
Platform: Linux-6.8.0-107-generic-x86_64-with-glibc2.35
Working directory: /workspace
torch: 2.5.1+cu124
CUDA available: True
CUDA runtime: 12.4
GPU 0: NVIDIA A100-SXM4-80GB (79.2 GB VRAM)
Thu Jun 11 15:47:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00

## 3. Imports


In [17]:
import inspect
import os
import sys
from pathlib import Path
from pprint import pprint
from typing import Any, Mapping

os.environ.setdefault("HF_HOME", "/workspace/.cache/huggingface")
os.environ.setdefault("TRANSFORMERS_CACHE", "/workspace/.cache/huggingface/transformers")
os.environ.setdefault("HF_DATASETS_CACHE", "/workspace/.cache/huggingface/datasets")
HF_TOKEN = os.environ.get("HF_TOKEN") or None

import yaml
import torch
from datasets import Dataset, DatasetDict, load_dataset
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForLanguageModeling, Trainer, TrainingArguments, set_seed

try:
    import wandb
except ModuleNotFoundError:
    wandb = None

print("imports complete")
print(f"HF_HOME={os.environ['HF_HOME']}")


imports complete
HF_HOME=/workspace/.cache/huggingface


## 4. Config

Edit this dictionary directly in Jupyter. The default values are full training settings for A100 80GB.


In [ ]:
CONFIG: dict[str, Any] = {
    "model_name": "Qwen/Qwen2.5-Coder-14B-Instruct",
    "dataset_name": "hitoshura25/crossvul",
    "dataset_split": "train",
    "eval_split": None,
    "output_dir": "/workspace/checkpoints/defapi-qwen2.5-coder-14b-lora",
    "max_seq_length": 2048,
    "attn_implementation": "sdpa",
    "gradient_checkpointing": False,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "learning_rate": 2e-4,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "save_steps": 200,
    "eval_steps": 200,
    "logging_steps": 10,
    "save_total_limit": 2,
    "report_to": "wandb",
    "wandb_project": "defapi-finetuning",
    "wandb_run_name": "qwen2.5-coder-14b-lora-full",
    "seed": 42,
}

set_seed(int(CONFIG["seed"]))

Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)
pprint(CONFIG)


## 5. Hugging Face / W&B Login Guide

For private or gated Hugging Face datasets/models, run `huggingface-cli login` in a Jupyter terminal or use `notebook_login()`. W&B login runs only when `CONFIG["report_to"] == "wandb"`.


In [19]:
# Hugging Face login options:
# 1. Jupyter Terminal: huggingface-cli login
# 2. Notebook:
# from huggingface_hub import notebook_login
# notebook_login()

if CONFIG["report_to"] == "wandb":
    if wandb is None:
        raise ModuleNotFoundError("wandb is not installed. Run the locked package install cell or set report_to='none'.")
    os.environ.setdefault("WANDB_PROJECT", CONFIG["wandb_project"])
    wandb.login()
else:
    os.environ["WANDB_DISABLED"] = "true"
    print("W&B disabled because CONFIG['report_to'] is 'none'.")


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


## 6. Dataset Load

Supports Hugging Face split slicing such as `train[:100]`. If no eval split is configured, the notebook creates an eval split with `train_test_split(test_size=0.1)`. The default dataset is `hitoshura25/crossvul`.


In [20]:
def load_defapi_dataset(config: Mapping[str, Any]) -> DatasetDict:
    dataset_name = str(config["dataset_name"])
    if dataset_name.startswith("<"):
        raise ValueError("Set CONFIG['dataset_name'] to a real Hugging Face dataset name before running this cell.")

    train_dataset = load_dataset(dataset_name, split=config["dataset_split"], token=HF_TOKEN)
    eval_split = config.get("eval_split")

    if eval_split:
        eval_dataset = load_dataset(dataset_name, split=eval_split, token=HF_TOKEN)
        return DatasetDict({"train": train_dataset, "eval": eval_dataset})

    if len(train_dataset) < 2:
        raise ValueError("At least 2 rows are required to auto-create an eval split.")

    split = train_dataset.train_test_split(test_size=0.1, seed=int(config["seed"]), shuffle=True)
    return DatasetDict({"train": split["train"], "eval": split["test"]})


dataset = load_defapi_dataset(CONFIG)
print(dataset)
print("train columns:", dataset["train"].column_names)
print("eval columns:", dataset["eval"].column_names)


DatasetDict({
    train: Dataset({
        features: ['cwe_id', 'cwe_description', 'language', 'vulnerable_code', 'fixed_code', 'file_pair_id', 'source', 'language_dir'],
        num_rows: 8381
    })
    eval: Dataset({
        features: ['cwe_id', 'cwe_description', 'language', 'vulnerable_code', 'fixed_code', 'file_pair_id', 'source', 'language_dir'],
        num_rows: 932
    })
})
train columns: ['cwe_id', 'cwe_description', 'language', 'vulnerable_code', 'fixed_code', 'file_pair_id', 'source', 'language_dir']
eval columns: ['cwe_id', 'cwe_description', 'language', 'vulnerable_code', 'fixed_code', 'file_pair_id', 'source', 'language_dir']


## 7. Dataset Validation

The notebook supports DefAPI `instruction`/`input`/`output` rows and CrossVul rows with `cwe_id`, `cwe_description`, `language`, `vulnerable_code`, and `fixed_code`.


In [21]:
REQUIRED_FIELDS = ("instruction", "input", "output")
CROSSVUL_FIELDS = ("cwe_id", "cwe_description", "language", "vulnerable_code", "fixed_code")


def has_fields(example: Mapping[str, Any], fields: tuple[str, ...]) -> bool:
    return all(field in example for field in fields)


def normalize_text(value: Any) -> str:
    return str(value).strip()


def convert_example(example: Mapping[str, Any]) -> dict[str, str]:
    """Convert one dataset row into instruction/input/output strings."""
    if has_fields(example, REQUIRED_FIELDS):
        converted = {field: normalize_text(example[field]) for field in REQUIRED_FIELDS}
    elif has_fields(example, CROSSVUL_FIELDS):
        cwe_id = normalize_text(example["cwe_id"])
        cwe_description = normalize_text(example["cwe_description"])
        language = normalize_text(example["language"])
        vulnerable_code = normalize_text(example["vulnerable_code"])
        fixed_code = normalize_text(example["fixed_code"])
        language_tag = language.lower().replace(" ", "-") or "text"
        converted = {
            "instruction": f"{language} 코드의 {cwe_id} 취약점을 분석하고 안전하게 수정하라.",
            "input": (
                f"CWE: {cwe_id}\n"
                f"Description: {cwe_description}\n"
                f"Language: {language}\n\n"
                f"Vulnerable code:\n```{language_tag}\n{vulnerable_code}\n```"
            ),
            "output": (
                f"1. 취약점 설명\n{cwe_description}\n\n"
                f"2. 안전한 수정 코드\n```{language_tag}\n{fixed_code}\n```\n\n"
                "3. 수정 이유\n"
                "보안 민감 경로에 위험한 입력이 직접 전달되지 않도록 안전한 API와 검증 흐름을 사용한다.\n\n"
                "4. 추가 주의사항\n"
                "동일한 데이터 흐름의 호출부와 테스트 케이스도 함께 점검한다."
            ),
        }
    else:
        available = ", ".join(example.keys())
        raise ValueError(
            "Dataset row must contain either instruction/input/output fields "
            f"or CrossVul fields. Available columns: {available}."
        )

    empty = [field for field, value in converted.items() if not value]
    if empty:
        raise ValueError(f"Dataset row has empty required fields: {empty}")
    return converted


def validate_dataset(dataset_dict: DatasetDict) -> None:
    for split_name, split_dataset in dataset_dict.items():
        if len(split_dataset) == 0:
            raise ValueError(f"{split_name} split is empty.")
        convert_example(split_dataset[0])
        print(f"{split_name}: {len(split_dataset)} rows validated against convert_example().")


validate_dataset(dataset)


train: 8381 rows validated against convert_example().
eval: 932 rows validated against convert_example().


## 8. Prompt Formatting

Creates a `text` field using the Qwen chat template when available. A fallback string formatter is included for debugging.


In [22]:
SYSTEM_MESSAGE = "You are a secure coding assistant. Analyze vulnerable code and provide safe, practical fixes."
RESPONSE_FORMAT = """응답은 반드시 다음 네 섹션을 포함해야 한다.
1. 취약점 설명
2. 안전한 수정 코드
3. 수정 이유
4. 추가 주의사항"""


def build_messages(example: Mapping[str, Any]) -> list[dict[str, str]]:
    converted = convert_example(example)
    user_content = (
        f"작업:\n{converted['instruction']}\n\n"
        f"출력 형식:\n{RESPONSE_FORMAT}\n\n"
        f"분석 대상:\n{converted['input']}"
    )
    return [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": converted["output"]},
    ]


def fallback_format(example: Mapping[str, Any]) -> str:
    messages = build_messages(example)
    return "\n".join(f"<{message['role']}>\n{message['content']}" for message in messages)


print("fallback formatted sample:")
print(fallback_format(dataset["train"][0])[:2000])


fallback formatted sample:
<system>
You are a secure coding assistant. Analyze vulnerable code and provide safe, practical fixes.
<user>
작업:
python 코드의 CWE-522 취약점을 분석하고 안전하게 수정하라.

출력 형식:
응답은 반드시 다음 네 섹션을 포함해야 한다.
1. 취약점 설명
2. 안전한 수정 코드
3. 수정 이유
4. 추가 주의사항

분석 대상:
CWE: CWE-522
Description: Insufficiently Protected Credentials - The product transmits or stores authentication credentials, but it uses an insecure method that is susceptible to unauthorized interception and/or retrieval.
Language: python

Vulnerable code:
```python
# -*- coding: utf-8 -*-

"""
requests.session
~~~~~~~~~~~~~~~~

This module provides a Session object to manage and persist settings across
requests (cookies, auth, proxies).
"""
import os
import sys
import time
from datetime import timedelta

from .auth import _basic_auth_str
from .compat import cookielib, is_py3, OrderedDict, urljoin, urlparse, Mapping
from .cookies import (
    cookiejar_from_dict, extract_cookies_to_jar, RequestsCookieJar, merge_cookies)
fro

## 9. Tokenizer Load


In [23]:
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    trust_remote_code=True,
    token=HF_TOKEN,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"
print(f"pad_token={tokenizer.pad_token!r}, eos_token={tokenizer.eos_token!r}, padding_side={tokenizer.padding_side}")


def format_with_tokenizer(example: Mapping[str, Any]) -> dict[str, str]:
    messages = build_messages(example)
    if hasattr(tokenizer, "apply_chat_template"):
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    else:
        text = fallback_format(example)
    return {"text": text}


formatted_dataset = dataset.map(
    format_with_tokenizer,
    remove_columns=dataset["train"].column_names,
    desc="Formatting DefAPI chat samples",
)

print("formatted sample:")
print(formatted_dataset["train"][0]["text"][:2000])


def tokenize_text_batch(batch: Mapping[str, list[str]]) -> dict[str, list[list[int]]]:
    return tokenizer(
        batch["text"],
        max_length=int(CONFIG["max_seq_length"]),
        padding=False,
        truncation=True,
    )


tokenized_dataset = formatted_dataset.map(
    tokenize_text_batch,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing DefAPI chat samples",
)

print("tokenized columns:", tokenized_dataset["train"].column_names)
print("first tokenized length:", len(tokenized_dataset["train"][0]["input_ids"]))


pad_token='<|endoftext|>', eos_token='<|im_end|>', padding_side=right
formatted sample:
<|im_start|>system
You are a secure coding assistant. Analyze vulnerable code and provide safe, practical fixes.<|im_end|>
<|im_start|>user
작업:
python 코드의 CWE-522 취약점을 분석하고 안전하게 수정하라.

출력 형식:
응답은 반드시 다음 네 섹션을 포함해야 한다.
1. 취약점 설명
2. 안전한 수정 코드
3. 수정 이유
4. 추가 주의사항

분석 대상:
CWE: CWE-522
Description: Insufficiently Protected Credentials - The product transmits or stores authentication credentials, but it uses an insecure method that is susceptible to unauthorized interception and/or retrieval.
Language: python

Vulnerable code:
```python
# -*- coding: utf-8 -*-

"""
requests.session
~~~~~~~~~~~~~~~~

This module provides a Session object to manage and persist settings across
requests (cookies, auth, proxies).
"""
import os
import sys
import time
from datetime import timedelta

from .auth import _basic_auth_str
from .compat import cookielib, is_py3, OrderedDict, urljoin, urlparse, Mapping
from .cookies impo

## 10. Model Load


In [ ]:
model_load_kwargs: dict[str, Any] = {
    "device_map": "auto",
    "torch_dtype": torch.bfloat16,
    "trust_remote_code": True,
    "attn_implementation": CONFIG["attn_implementation"],
    "token": HF_TOKEN,
}

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    **model_load_kwargs,
)
model.config.use_cache = False
print(f"model loaded with attention: {model.config._attn_implementation}")


## 11. LoRA Config


In [25]:
lora_config = LoraConfig(
    r=int(CONFIG["lora_r"]),
    lora_alpha=int(CONFIG["lora_alpha"]),
    lora_dropout=float(CONFIG["lora_dropout"]),
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
lora_config


trainable params: 68,812,800 || all params: 14,838,846,464 || trainable%: 0.4637


LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path='Qwen/Qwen2.5-Coder-14B-Instruct', revision=None, task_type='CAUSAL_LM', inference_mode=False, r=16, target_modules={'q_proj', 'v_proj', 'gate_proj', 'k_proj', 'o_proj', 'down_proj', 'up_proj'}, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False))

## 12. TrainingArguments

Handles `eval_strategy` vs `evaluation_strategy` across Transformers versions. The notebook tokenizes the `text` column explicitly before training, so the trainer receives tensor-ready `input_ids` and `attention_mask` columns.


In [ ]:
def supports_parameter(cls: type, parameter_name: str) -> bool:
    return parameter_name in inspect.signature(cls.__init__).parameters


def supported_init_kwargs(cls: type, kwargs: dict[str, Any]) -> dict[str, Any]:
    signature = inspect.signature(cls.__init__)
    if any(parameter.kind == inspect.Parameter.VAR_KEYWORD for parameter in signature.parameters.values()):
        return kwargs
    return {key: value for key, value in kwargs.items() if key in signature.parameters}

training_kwargs: dict[str, Any] = {
    "output_dir": CONFIG["output_dir"],
    "run_name": CONFIG["wandb_run_name"],
    "report_to": CONFIG["report_to"],
    "num_train_epochs": CONFIG["num_train_epochs"],
    "per_device_train_batch_size": CONFIG["per_device_train_batch_size"],
    "per_device_eval_batch_size": CONFIG["per_device_train_batch_size"],
    "gradient_accumulation_steps": CONFIG["gradient_accumulation_steps"],
    "learning_rate": CONFIG["learning_rate"],
    "warmup_ratio": 0.03,
    "lr_scheduler_type": "cosine",
    "logging_steps": CONFIG["logging_steps"],
    "eval_steps": CONFIG["eval_steps"],
    "save_steps": CONFIG["save_steps"],
    "save_total_limit": CONFIG["save_total_limit"],
    "bf16": True,
    "gradient_checkpointing": CONFIG["gradient_checkpointing"],
    "optim": "adamw_torch",
    "save_strategy": "steps",
    "logging_strategy": "steps",
    "remove_unused_columns": False,
}


strategy_name = "eval_strategy" if supports_parameter(TrainingArguments, "eval_strategy") else "evaluation_strategy"
training_kwargs[strategy_name] = "steps"

training_args = TrainingArguments(**supported_init_kwargs(TrainingArguments, training_kwargs))
training_args


## 13. Trainer

Uses a `transformers.Trainer` subclass that does not call `.to(device)`, because the model is already dispatched by `device_map="auto"`. The LoRA adapter is attached before trainer construction.


In [27]:
class NoMoveTrainer(Trainer):
    def _move_model_to_device(self, model, device):
        return model


data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = NoMoveTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["eval"],
    data_collator=data_collator,
)
trainer


## 14. Full Training Cell

Runs the full training config, saves the adapter/model state and tokenizer, and prints the checkpoint path.


In [ ]:
torch.cuda.empty_cache()
train_result = trainer.train()

output_dir = Path(CONFIG["output_dir"])
trainer.save_model(str(output_dir))
tokenizer.save_pretrained(str(output_dir))

print(train_result)
print(f"Adapter/model and tokenizer saved to: {output_dir}")


## 15. Inference Test Cell

Loads the fine-tuned adapter and generates a short response for a safe command-injection example.


In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    **model_load_kwargs,
)
inference_model = PeftModel.from_pretrained(base_model, CONFIG["output_dir"])
inference_model.eval()

example_instruction = "다음 코드의 보안 취약점을 분석하고 안전한 코드로 수정하라."
example_input = """Python code:\n```python\nimport os\n\ndef run(user_input):\n    os.system('grep ' + user_input + ' /var/log/app.log')\n```"""
messages = [
    {"role": "system", "content": SYSTEM_MESSAGE},
    {
        "role": "user",
        "content": f"작업:\n{example_instruction}\n\n출력 형식:\n{RESPONSE_FORMAT}\n\n분석 대상:\n{example_input}",
    },
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(inference_model.device)
with torch.no_grad():
    output_ids = inference_model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
response = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print(response)

for marker in ["1.", "2.", "3.", "4."]:
    print(f"contains {marker}: {marker in response}")


## 16. Full Training Config Cell

These values are already the notebook defaults; keep this cell as a reference if you switch back from a smoke run.


In [ ]:
FULL_TRAINING_CONFIG = {
    **CONFIG,
    "dataset_split": "train",
    "output_dir": "/workspace/checkpoints/defapi-qwen2.5-coder-14b-lora",
    "max_seq_length": 2048,
    "attn_implementation": "sdpa",
    "gradient_checkpointing": False,
    "gradient_accumulation_steps": 16,
    "save_steps": 200,
    "eval_steps": 200,
    "logging_steps": 10,
    "num_train_epochs": 1,  # use 2 only after confirming quality/cost
    "report_to": "wandb",
}

pprint(FULL_TRAINING_CONFIG)


## 17. Troubleshooting

- **CUDA driver mismatch**: keep the RunPod CUDA image and PyTorch CUDA wheel aligned, for example cu124 images with a cu124 PyTorch wheel.
- **Transformers / float8 errors**: if very new `transformers` builds fail with torch float8 symbols such as `float8_e8m0fnu`, use a stable combo such as `transformers==4.46.3`.
- **A100 LoRA OOM**: reduce `max_seq_length`, increase `gradient_accumulation_steps`, or set `CONFIG["gradient_checkpointing"] = True` if VRAM becomes tight.
- **Attention backend issues**: keep `CONFIG["attn_implementation"] = "sdpa"` unless FlashAttention is installed and verified in the runtime.
- **General OOM**: reduce `max_seq_length`, increase `gradient_accumulation_steps`, and use the smoke config until the first pass succeeds.
- **Disable W&B**: set `CONFIG["report_to"] = "none"`.
- **Private Hugging Face datasets/models**: run `huggingface-cli login` in a Jupyter Terminal before loading.
- **Workspace paths**: use `/workspace` for caches and checkpoints, not `/root`.

Execution order: run sections 1-13 to prepare, run section 14 for full training, then run section 15 for inference verification.
